# 手順2-2：マスタデータ作成およびデータ準備

このノートブックでは、CSVファイルからSilver層のマスターテーブルを作成し、取り込むCSVデータを準備ます。

## 作成するテーブル
1. **silver_schema.customer_master** - 顧客マスター
2. **silver_schema.contract_master** - 契約マスター

## データソース
- `/Workspace/Users/{user}/lakeflow_handson/handson_codes/demo_data/customer_master.csv`
- `/Workspace/Users/{user}/lakeflow_handson/handson_codes/demo_data/contract_master.csv`

## CSVデータの配置
- **telematics_data_20260801.csv**

## 配置先
- `bronze_schema/data_source/`

In [0]:
# 環境設定

# 現在のユーザー名を取得（CSVパス用）
current_user = spark.sql("SELECT current_user() as user").collect()[0][0]

# カタログとスキーマの設定（セットアップノートブックと一致させる）
catalog_name = "lakeflow_training"
silver_schema = "silver_schema"

print(f"カタログ: {catalog_name}")
print(f"Silver層スキーマ: {silver_schema}")
print(f"作成先: {catalog_name}.{silver_schema}")
print(f"CSVパス用ユーザー: {current_user}")

In [0]:
# CSVファイルから顧客マスターテーブルを作成
# ユーザー情報から動的にパスを構築
customer_csv_path = f"/Workspace/Users/{current_user}/lakeflow_handson/handson_codes/demo_data/customer_master.csv"

# CSVを読み込み
df_customer = spark.read.csv(
    customer_csv_path,
    header=True,
    inferSchema=True
)

# テーブルとして保存
table_name = f"{catalog_name}.{silver_schema}.customer_master"
df_customer.write.mode("overwrite").saveAsTable(table_name)

print(f"✓ 顧客マスターテーブルを作成しました: {table_name}")
print(f"  レコード数: {df_customer.count()}件")

# データサンプル表示
print("\n【データサンプル】")
display(df_customer.limit(5))

In [0]:
# CSVファイルから契約マスターテーブルを作成
# ユーザー情報から動的にパスを構築
contract_csv_path = f"/Workspace/Users/{current_user}/lakeflow_handson/handson_codes/demo_data/contract_master.csv"

# CSVを読み込み
df_contract = spark.read.csv(
    contract_csv_path,
    header=True,
    inferSchema=True
)

# テーブルとして保存
table_name = f"{catalog_name}.{silver_schema}.contract_master"
df_contract.write.mode("overwrite").saveAsTable(table_name)

print(f"✓ 契約マスターテーブルを作成しました: {table_name}")
print(f"  レコード数: {df_contract.count()}件")

# データサンプル表示
print("\n【データサンプル】")
display(df_contract.limit(5))

In [0]:
# 作成したテーブルの確認
print("="*60)
print("マスターテーブル作成完了")
print("="*60)

# テーブル一覧を表示
print(f"\n【{catalog_name}.{silver_schema} のテーブル一覧】")
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{silver_schema}"))

# 各テーブルの詳細情報
print("\n【顧客マスターテーブル情報】")
spark.sql(f"DESCRIBE TABLE {catalog_name}.{silver_schema}.customer_master").show(truncate=False)

print("\n【契約マスターテーブル情報】")
spark.sql(f"DESCRIBE TABLE {catalog_name}.{silver_schema}.contract_master").show(truncate=False)

In [0]:
# テレマティクスデータをsource_data Volumeにコピー
import os

# ソースファイルパス（ユーザーのワークスペース内）
source_csv = f"/Workspace/Users/{current_user}/lakeflow_handson/handson_codes/demo_data/telematics_data_20260801.csv"

# コピー先Volume（Lakeflowパイプラインの取り込み元として使用）
volume_path = f"/Volumes/{catalog_name}/bronze_schema/source_data"
target_csv = f"{volume_path}/telematics_data_20260801.csv"

print(f"ソース: {source_csv}")
print(f"コピー先: {target_csv}")
print()

# ファイルをVolumeにコピー
try:
    dbutils.fs.cp(source_csv, target_csv, recurse=False)
    print("✓ ファイルのコピーが完了しました")
    
    # Volume内のファイル一覧を確認
    print(f"\n【{volume_path} 内のファイル一覧】")
    files = dbutils.fs.ls(volume_path)
    for file in files:
        print(f"  - {file.name} ({file.size:,} bytes)")
        
except Exception as e:
    print(f"エラー: {e}")